# Lesson 23 — Generalized Disjunctive Programming

## Learning objectives

1. Write a GDP with disjunctions over algebraic constraints.
2. Convert GDP to MINLP via big-M and convex-hull reformulations.
3. Recognize when convex hull is worth the extra variables.
4. Use `discopt`'s GDP frontend (or build one).

## 1. Disjunctive programming

A *disjunction* is an "or": at least one branch must hold. Standard form {cite:p}`Raman1994,Grossmann2013`:

$$
\bigvee_{k \in K} \begin{bmatrix} Y_k \\ A_k x \le b_k \\ f_k(x) \le 0 \end{bmatrix}
$$

with $Y_k$ a Boolean variable. Exactly one $Y_k$ is True. The constraints in branch $k$ are active iff $Y_k$ is True.

Logical constraints between $Y_k$'s can be expressed (e.g., $Y_1 \lor Y_2 \Rightarrow \neg Y_3$).

GDP is *the* natural framework for process synthesis problems with discrete unit-selection decisions {cite:p}`Grossmann2013,Trespalacios2015`.

## 2. Big-M reformulation

Convert each disjunctive constraint $A_k x \le b_k$ to

$$A_k x \le b_k + M_k (1 - y_k)$$

with binary $y_k$ corresponding to $Y_k$, and $\sum_k y_k = 1$. Plus indicators on $f_k(x) \le 0$.

**Pros:** simple, smaller model.
**Cons:** weak relaxation; tightness depends on $M_k$ {cite:p}`Sawaya2005`.

## 3. Convex-hull reformulation

For each $k$, introduce a copy $x_k$ of the variables. Let $x = \sum_k x_k$. Constrain $A_k x_k \le b_k y_k$ and $\sum y_k = 1$. The convex hull of the disjunction is then:

$$x = \sum_k x_k, \quad A_k x_k \le b_k y_k, \quad x_k \in y_k X_k, \quad \sum y_k = 1, \; y \ge 0.$$

For nonlinear $f_k$, use perspective functions: $y_k f_k(x_k / y_k)$ {cite:p}`Trespalacios2015`.

**Pros:** tightest possible LP relaxation.
**Cons:** $|K|$ copies of variables (model size grows). Worth it when big-M is loose.

## 4. Practical recipe

Default: big-M with carefully derived $M$. Switch to convex hull when:

- big-M values are "large" (no natural physical bound),
- the problem is solving slowly with much branching on indicators,
- the disjunction has a small number of branches ($|K| \le 3$).

`discopt` provides both; flag `gdp_reformulation='hull'` vs `'bigm'` {cite:p}`Sawaya2005`.

In [ ]:
# discopt course compat shim — installs `add_variable / add_variables /
# add_constraint / add_constraints / is_convex` and a friendly `.solve(...)`
# on `discopt.Model`. Run this cell first.
import sys, pathlib
_repo = pathlib.Path.cwd()
while _repo != _repo.parent and not (_repo / "course").is_dir(): _repo = _repo.parent
if str(_repo) not in sys.path: sys.path.insert(0, str(_repo))
from course._compat import *  # noqa: F401,F403


In [ ]:
import numpy as np, discopt as do
from discopt.modeling import Model

# Process synthesis: pick reactor type (CSTR or PFR) for one stage
# CSTR: cost 100 + 5*F   (F flow)
# PFR : cost 60  + 8*F + 0.5*F^2
# Production target F = 4. Minimize cost.

m = Model("reactor")
F = m.add_variable(lb=0, ub=10)
m.add_constraint(F == 4)

cost = m.add_disjunction([
    ("CSTR", 100 + 5 * F),
    ("PFR",   60 + 8 * F + 0.5 * F**2),
])  # exactly one branch active
m.minimize(cost)
r = m.solve()
print("chosen branch:", r.disjunction_choice("CSTR_or_PFR"))
print("cost          :", r.objective)


## 5. Beyond two-branch disjunctions

Multi-branch disjunctions (3 or more options) and *nested* disjunctions are common in process synthesis. The convex-hull reformulation generalizes; big-M does too but loses tightness.

Logical constraints between disjunctions (e.g., "if reactor type A is chosen at stage 1 then separator type X cannot be chosen at stage 2") are easy to express as Boolean implications and reformulate to linear constraints on $y$.

## References
{cite:p}`Raman1994,Grossmann2013,Trespalacios2015,Sawaya2005`.